In [1]:
%load_ext autoreload
%autoreload 2

In [28]:
import pandas as pd
import numpy as np
import pyaging as pya
import torch
from art.estimators.regression.pytorch import PyTorchRegressor
import pathlib
from scipy.stats import iqr, pearsonr
from art.attacks.evasion import BasicIterativeMethod, ProjectedGradientDescent
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error

In [3]:
norm = "BMIQ"
path = "E:/YandexDisk/bbd/attacks_on_clocks"
path_clock_data = "E:/YandexDisk/pydnameth/datasets/pyaging"

pheno = pd.read_csv(f"{path}/data/pheno.csv", index_col=0)
betas = pd.read_pickle(f"{path}/data/betas{norm}.pkl")

feats_pheno = ['Age', 'Sex', 'Tissue']
pheno = pheno[feats_pheno]

df_clocks = pd.merge(pheno, betas, left_index=True, right_index=True)

In [ ]:
clocks_names = ["hannum"]

#clock_names_file = open(f"{path}/clocks_list.txt", "r") 
#clocks_names = clock_names_file.read().split("\n") 
#clock_names_file.close() 

epsilons = sorted(list(set.union(
    set(np.linspace(0.00000001, 0.0000001, 10)), 
    set(np.linspace(0.0000001, 0.000001, 10)),
    set(np.linspace(0.000001, 0.00001, 10)),
    set(np.linspace(0.00001, 0.0001, 10)),
    set(np.linspace(0.0001, 0.001, 10)),
    set(np.linspace(0.001, 0.01, 10)),
    set(np.linspace(0.01, 0.1, 10)),
    set(np.linspace(0.1, 1.0, 10)),
)))

for clock_name in clocks_names:

    path_clock = f"{path}/result/{clock_name}"
    pathlib.Path(f"{path_clock}/Evasion").mkdir(parents=True, exist_ok=True)

    col_trgt = 'Age'
    col_pred = clock_name

    adata = pya.pp.df_to_adata(df_clocks, metadata_cols=['Sex', 'Tissue'], imputer_strategy='knn', verbose=True)
    pya.pred.predict_age(adata=adata, dir=path_clock_data, clock_names=clock_name, verbose=True)
    results = pd.merge(pheno.loc[:, feats_pheno], adata.obs[clock_name], left_index=True, right_index=True)

    logger = pya.logger.Logger('test_logger')
    device = 'cpu'
    indent_level = 1

    clock = pya.pred.load_clock(clock_name, device, path_clock_data, logger, indent_level=indent_level)
    clock_features = clock.features
    clock_reference_values = clock.reference_values 

    common_cpgs = list(set(clock_features).intersection(betas.columns))
    differ_cpgs = list(set(clock_features).difference(betas.columns))
    missing_indices = [clock_features.index(curr_cpg) for curr_cpg in differ_cpgs]
    if clock_reference_values is not None:
        missing_references = [clock_reference_values[ref] for ref in missing_indices]
    else:
        mean_df = np.mean(betas.loc[:, common_cpgs])
        missing_references = [mean_df for ref in missing_indices]
    df = betas.loc[:, common_cpgs]
    df[differ_cpgs] = pd.DataFrame([missing_references], index=df.index)
    df = pd.merge(df, results, left_index=True, right_index=True)
    df[f"{clock_name}_MAE"] = df[col_pred] - df[col_trgt]
    df[f"{clock_name}_MAE_abs"] = df[f"{clock_name}_MAE"].abs()
    
    linreg = smf.ols(formula=f"{clock_name} ~ Age", data=df).fit()
    df[f"{clock_name}_linear_pred"] = linreg.predict(df)
    df[f"{clock_name}_MAE_lin"] = df[col_pred] - df[f"{clock_name}_linear_pred"]
    df[f"{clock_name}_MAE_lin_abs"] = df[f"{clock_name}_MAE_lin"].abs()
    
    df.to_pickle(f"{path_clock}/Evasion/df_origin.pkl")
    
    model = clock
    for model_component in model._modules:
        if isinstance(getattr(model, model_component), pya.models._base_models.LinearModel):
            getattr(model, model_component).linear.weight = torch.nn.parameter.Parameter(getattr(model, model_component).linear.weight.to(torch.float32))
            getattr(model, model_component).linear.bias = torch.nn.parameter.Parameter(getattr(model, model_component).linear.bias.to(torch.float32))
        if isinstance(getattr(model, model_component), pya.models._base_models.PCLinearModel):
            getattr(model, model_component).linear.weight = torch.nn.parameter.Parameter(getattr(model, model_component).linear.weight.to(torch.float32))
            getattr(model, model_component).linear.bias = torch.nn.parameter.Parameter(getattr(model, model_component).linear.bias.to(torch.float32))
            getattr(model, model_component).center = torch.nn.parameter.Parameter(getattr(model, model_component).center.to(torch.float32))
            getattr(model, model_component).rotation = torch.nn.parameter.Parameter(getattr(model, model_component).rotation.to(torch.float32))
        if isinstance(getattr(model, model_component), pya.models._base_models.AltumAgeNeuralNetwork):
            for curr_model_component in model.base_model._modules:
                if isinstance(getattr(model.base_model, curr_model_component), torch.nn.Linear):
                    getattr(model.base_model, curr_model_component).weight = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).weight.to(torch.float32))
                    getattr(model.base_model, curr_model_component).bias = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).bias.to(torch.float32))
                if isinstance(getattr(model.base_model, curr_model_component), torch.nn.BatchNorm1d):
                    getattr(model.base_model, curr_model_component).weight = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).weight.to(torch.float32))
                    getattr(model.base_model, curr_model_component).bias = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).bias.to(torch.float32))
                    getattr(model.base_model, curr_model_component).running_mean = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).running_mean.to(torch.float32))
                    getattr(model.base_model, curr_model_component).running_mean.requires_grad = False
                    getattr(model.base_model, curr_model_component).running_var = torch.nn.parameter.Parameter(getattr(model.base_model, curr_model_component).running_var.to(torch.float32))
                    getattr(model.base_model, curr_model_component).running_var.requires_grad = False

    art_regressor = PyTorchRegressor(
        model=model,
        loss=torch.nn.L1Loss(),
        input_shape=[len(clock_features)],
        use_amp=False,
        opt_level="O1",
        loss_scale="dynamic",
        channels_first=True,
        clip_values=None,
        preprocessing_defences=None,
        postprocessing_defences=None,
        preprocessing=(0.0, 1.0),
        device_type="cpu",
    )

    mae_diag = mean_absolute_error(df[col_pred].values, df[col_trgt].values)
    mae_regr = np.mean(np.abs(df[f"{clock_name}_MAE_lin"].values))
    rho, _ = pearsonr(df[col_pred].values, df[col_trgt].values)

    df_eps = pd.DataFrame(index=[0].extend(epsilons))
    df_eps.loc[0, f"MAE"] = mae_diag
    df_eps.loc[0, f"MAE_lin"] = mae_regr
    df_eps.loc[0, f"Pearson_rho"] = rho

    for eps_raw in epsilons:

        eps = np.array([eps_raw * iqr(df.loc[:, feat].values) for feat in clock_features])
        eps_step = np.array([0.2 * eps_raw * iqr(df.loc[:, feat].values) + 1e-6 for feat in clock_features])

        attacks = {
            'ProjectedGradientDescent': ProjectedGradientDescent(
                estimator=art_regressor,
                norm=np.inf,
                eps=eps,
                eps_step=eps_step,
                decay=None,
                max_iter=100,
                targeted=False,
                num_random_init=0,
                batch_size=512,
                random_eps=False,
                summary_writer=False,
                verbose=True
        ),
#            'BasicIterative': BasicIterativeMethod(
#                estimator=art_regressor,
#                eps=eps,
#                eps_step=eps_step,
#                max_iter=100,
#                targeted=False,
#                batch_size=512,
#                verbose=True
#            )
        }

        for attack_name, attack in attacks.items():
            path_curr = f"{path_clock}/Evasion/{attack_name}"
            pathlib.Path(f"{path_curr}").mkdir(parents=True, exist_ok=True)

            X_adv = attack.generate(df.loc[:, clock_features].values.astype(np.float32))
            
            df_adv = df.loc[:, [col_trgt]].copy()
            df_adv.loc[:, clock_features] = X_adv
            df_adv[col_pred] = model(torch.from_numpy(np.float32(df_adv.loc[:, clock_features].values))).cpu().detach().numpy().ravel()
            df_adv[f"{clock_name}_MAE"] = df_adv[col_pred] - df_adv[col_trgt]
            df_adv[f"{clock_name}_MAE_abs"] = df_adv[f"{clock_name}_MAE"].abs()

            curr_linreg = smf.ols(formula=f"{clock_name} ~ Age", data=df_adv).fit()
            df_adv[f"{clock_name}_linear_pred"] = curr_linreg.predict(df_adv)
            df_adv[f"{clock_name}_MAE_lin"] = df_adv[col_pred] - df_adv[f"{clock_name}_linear_pred"]
            df_adv[f"{clock_name}_MAE_lin_abs"] = df_adv[f"{clock_name}_MAE_lin"].abs()
                
            df_adv.to_pickle(f"{path_curr}/df_eps_{eps_raw:0.2e}.pkl")
            
            mae_diag = mean_absolute_error(df_adv[col_pred].values, df_adv[col_trgt].values)
            mae_regr = np.mean(np.abs(df_adv[f"{clock_name}_MAE_lin"].values))
            rho, _ = pearsonr(df_adv[col_pred].values, df_adv[col_trgt].values)
        
            df_eps.loc[eps_raw, f"MAE"] = mae_diag
            df_eps.loc[eps_raw, f"MAE_lin"] = mae_regr
            df_eps.loc[eps_raw, f"Pearson_rho"] = rho

    df_eps.to_excel(f"{path_clock}/Evasion/metrics_{attack_name}.xlsx", index_label='eps')

|-----> 🏗️ Starting df_to_adata function
|-----> ⚙️ Create anndata object started
|-----> ✅ Create anndata object finished [1.3188s]
|-----> ⚙️ Add metadata to anndata started
|-----------> Adding provided metadata to adata.obs
|-----> ✅ Add metadata to anndata finished [0.0018s]
|-----> ⚙️ Log data statistics started
|-----------> There are 729 observations
|-----------> There are 484913 features
|-----------> Total missing values: 0
|-----------> Percentage of missing values: 0.00%
|-----> ✅ Log data statistics finished [0.4472s]
|-----> ⚙️ Impute missing values started
|-----------> No missing values found. No imputation necessary
|-----> ✅ Impute missing values finished [0.4761s]
|-----> 🎉 Done! [8.8228s]
|-----> 🏗️ Starting predict_age function
|-----> ⚙️ Set PyTorch device started
|-----------> Using device: cpu
|-----> ✅ Set PyTorch device finished [0.0027s]
|-----> 🕒 Processing clock: hannum
|-----------> ⚙️ Load clock started
|-----------------> Data found in E:/YandexDisk/pyd

PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


...

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


PGD - Iterations:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([512, 1])) that is different to the input size (torch.Size([512])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\Users\alena\anaconda3\envs\gradiopy\lib\site-packages\torch\nn\modules\loss.py:101: UserWarning: Using a target size (torch.Size([217, 1])) that is different to the input size (torch.Size([217])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
